
# 🎯 Categorical Encoding in Machine Learning
### Professional Jupyter Notebook — Notes + Python + Interview Preparation

> **Goal:** Understand how categorical variables are converted into numerical representations, when to use each encoding technique, common mistakes, data leakage, and interview scenarios.

---

## 📌 Learning Roadmap

**Categorical Data → Encoding → Numerical Features → ML Model**

We will cover:

1. Label Encoding
2. Ordinal Encoding
3. One-Hot Encoding ⭐
4. Dummy Encoding
5. Frequency Encoding
6. Count Encoding
7. Target Encoding ⭐⭐⭐
8. Binary Encoding
9. Hash Encoding
10. Weight of Evidence (WoE)
11. High-cardinality features
12. Data leakage
13. `ColumnTransformer` + `Pipeline`
14. Encoding decision guide
15. Interview questions



## 🎨 1. What is Encoding?

**Encoding** is the process of converting categorical values into numerical representations that machine-learning algorithms can use.

### Example

| Gender | City | Education |
|---|---|---|
| Male | Pune | Graduate |
| Female | Mumbai | Postgraduate |
| Male | Nashik | Graduate |

A model cannot directly perform mathematical operations on strings such as `Male`, `Pune`, or `Graduate`.

### Basic flow

```text
Categorical Data
       ↓
    Encoding
       ↓
Numerical Data
       ↓
Machine Learning Model
```

### Two important types of categorical data

**Nominal:** No natural order  
Examples: City, Color, Gender

**Ordinal:** Meaningful order exists  
Examples: Low < Medium < High, School < Graduate < PhD


In [ ]:
# Install packages only if required.
# Uncomment the next line in a fresh environment.

# %pip install pandas scikit-learn category_encoders



# 2. Label Encoding

### Definition

Label Encoding assigns an integer to each category.

Example:

```text
B → 0
M → 1
```

It is especially common when encoding a **target variable**.

### ⚠️ Important

For nominal input features, arbitrary integer labels can create a false sense of order:

```text
Red → 0
Blue → 1
Green → 2
```

A model may treat `2 > 1 > 0`, even though colors have no natural ranking.


In [1]:
from sklearn.preprocessing import LabelEncoder

labels = ["B", "M", "B", "B", "M", "M"]

encoder = LabelEncoder()
encoded_labels = encoder.fit_transform(labels)

print("Original :", labels)
print("Encoded  :", encoded_labels)
print("Classes  :", encoder.classes_)


Original : ['B', 'M', 'B', 'B', 'M', 'M']
Encoded  : [0 1 0 0 1 1]
Classes  : ['B' 'M']



## Interview Point

**Q: When is Label Encoding commonly used?**

**Answer:** It is commonly used to convert a categorical target into numeric class labels. For nominal input features, One-Hot Encoding or another suitable technique is often safer because arbitrary integer ordering can be misleading.



# 3. Ordinal Encoding ⭐

Use **Ordinal Encoding** when categories have a meaningful order.

Example:

```text
Low        → 0
Medium     → 1
High       → 2
Very High  → 3
```

Here the numerical relationship represents a real ranking.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

df_ordinal = pd.DataFrame({
    "Satisfaction": ["Low", "High", "Medium", "Low", "Very High"]
})

order = [["Low", "Medium", "High", "Very High"]]

ordinal_encoder = OrdinalEncoder(categories=order)

df_ordinal["Satisfaction_Encoded"] = ordinal_encoder.fit_transform(
    df_ordinal[["Satisfaction"]]
)

df_ordinal



### Label vs Ordinal Encoding

| Feature | Label Encoding | Ordinal Encoding |
|---|---|---|
| Main idea | Assign integer labels | Preserve meaningful ranking |
| Natural order required | No | **Yes** |
| Typical use | Target/classes | Ordered input feature |
| Example | B/M | Low/Medium/High |



# 4. One-Hot Encoding ⭐⭐⭐

One-Hot Encoding creates a **separate binary column for each category**.

Example:

| City | Mumbai | Nashik | Pune |
|---|---:|---:|---:|
| Pune | 0 | 0 | 1 |
| Mumbai | 1 | 0 | 0 |
| Nashik | 0 | 1 | 0 |

`1` means the category is present.  
`0` means it is absent.

### Best suited for

**Nominal categorical variables** with a manageable number of unique categories.


In [ ]:
import pandas as pd

df_city = pd.DataFrame({
    "City": ["Pune", "Mumbai", "Nashik", "Pune", "Mumbai"]
})

one_hot = pd.get_dummies(df_city, dtype=int)

print("Original data:")
display(df_city)

print("One-Hot Encoded data:")
display(one_hot)



## One-Hot Encoding with Scikit-Learn

`handle_unknown="ignore"` is important in production pipelines because new/unseen categories can appear during prediction.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded = encoder.fit_transform(df_city[["City"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["City"])
)

encoded_df



# 5. Dummy Encoding

Dummy Encoding is closely related to One-Hot Encoding, but one category is commonly removed.

For 3 categories:

**Full One-Hot:**

```text
Red   → [1, 0, 0]
Blue  → [0, 1, 0]
Green → [0, 0, 1]
```

**Drop-first / Dummy representation:**

```text
Red   → [0, 0]
Blue  → [1, 0]
Green → [0, 1]
```

The omitted category becomes the **reference category**.


In [ ]:
dummy_encoded = pd.get_dummies(
    df_city,
    drop_first=True,
    dtype=int
)

dummy_encoded



## Dummy Variable Trap

For a linear model, if all one-hot columns are included, one column can be perfectly determined from the others.

Example:

```text
Female = 1 - Male
```

This can create **perfect multicollinearity**.

A common solution for linear models is:

```python
OneHotEncoder(drop="first")
```

### Important nuance

Dropping one category is not a universal requirement for every model. Tree-based models generally do not have the same multicollinearity concern as ordinary linear regression.



# 6. Frequency Encoding

Frequency Encoding replaces each category with how often it occurs.

Example:

```text
Pune    → 3 occurrences
Mumbai  → 2 occurrences
Nashik  → 1 occurrence
```

This is useful when a categorical feature has many unique categories.


In [ ]:
df_freq = pd.DataFrame({
    "City": ["Pune", "Pune", "Pune", "Mumbai", "Mumbai", "Nashik"]
})

frequency = df_freq["City"].value_counts()

df_freq["City_Frequency"] = df_freq["City"].map(frequency)

df_freq



# 7. Count Encoding

Count Encoding uses the **number of occurrences** of each category.

For the same data:

```text
Pune    → 3
Mumbai  → 2
Nashik  → 1
```

### Frequency vs Count

**Count:**

```text
count(category)
```

**Frequency:**

```text
count(category) / total_rows
```


In [ ]:
total_rows = len(df_freq)

df_freq["City_Frequency_Ratio"] = (
    df_freq["City_Frequency"] / total_rows
)

df_freq



# 8. Target Encoding ⭐⭐⭐

Target Encoding replaces a category with a statistic calculated from the **target variable**.

For binary classification, a common choice is the **mean target value**.

Example:

| City | Purchased |
|---|---:|
| Pune | 1 |
| Pune | 1 |
| Pune | 0 |
| Mumbai | 0 |
| Mumbai | 0 |
| Nashik | 1 |

Target mean:

```text
Pune   → (1 + 1 + 0) / 3 = 0.667
Mumbai → (0 + 0) / 2 = 0.000
Nashik → 1 / 1 = 1.000
```

### ⚠️ Major risk: Data Leakage

Never calculate target encoding using information from the validation/test set.

Correct conceptual flow:

```text
Dataset
   ↓
Train / Validation / Test Split
   ↓
Fit encoder using training data
   ↓
Transform training data
   ↓
Transform validation/test using learned mapping
```


In [ ]:
df_target = pd.DataFrame({
    "City": ["Pune", "Pune", "Pune", "Mumbai", "Mumbai", "Nashik"],
    "Purchased": [1, 1, 0, 0, 0, 1]
})

target_mean = df_target.groupby("City")["Purchased"].mean()

df_target["City_Target_Encoded"] = df_target["City"].map(target_mean)

print("Category target means:")
display(target_mean.to_frame("Target_Mean"))

print("Encoded data:")
display(df_target)



# 9. Binary Encoding

Binary Encoding first assigns integer codes and then represents those codes in binary form.

For example:

```text
Category A → 0 → 00
Category B → 1 → 01
Category C → 2 → 10
Category D → 3 → 11
```

### Why use it?

For high-cardinality features, it can use far fewer columns than One-Hot Encoding.

For example, thousands of categories can make One-Hot Encoding extremely wide.



# 10. Hash Encoding

Hash Encoding maps categories into a **fixed number of output columns** using a hash function.

### Advantages

- Useful for very high-cardinality data
- Fixed output dimensionality
- Can reduce memory requirements

### Disadvantage

Different categories can map to the same location.

This is called a **hash collision**.



# 11. Base-N Encoding

Base-N Encoding represents category codes using a base-N number system.

It is another dimensionality-reduction approach for high-cardinality categorical variables.

Examples:

```text
Base 2 → binary representation
Base 3 → ternary representation
Base 10 → decimal representation
```



# 12. Weight of Evidence (WoE)

WoE is widely associated with:

- Banking
- Credit scoring
- Risk modeling
- Logistic regression

A common conceptual formula is:

\[
WoE = \ln\left(\frac{Distribution\ of\ Good}{Distribution\ of\ Bad}\right)
\]

WoE is an advanced encoding method and is especially useful when interpretability and risk modeling are important.



# 13. Nominal vs Ordinal — The Most Important Decision

## Nominal

There is **no meaningful order**.

Examples:

```text
City: Pune, Mumbai, Delhi
Color: Red, Blue, Green
Department: HR, Sales, IT
```

Typical choice:

**One-Hot Encoding**

---

## Ordinal

There is a **meaningful order**.

Examples:

```text
Low < Medium < High
School < Graduate < Master < PhD
```

Typical choice:

**Ordinal Encoding**



# 14. High Cardinality

**Cardinality = number of unique categories.**

Example:

```text
City column:
Pune
Mumbai
Delhi
...
```

If there are 10,000 unique cities:

```text
One-Hot → ~10,000 columns
```

Possible alternatives:

- Frequency Encoding
- Count Encoding
- Target Encoding
- Binary Encoding
- Hash Encoding
- Rare-category grouping

### Interview rule

Do not automatically choose One-Hot Encoding. Consider **cardinality + model + dataset size + business context**.



# 15. Professional ML Pipeline ⭐⭐⭐

Encoding should generally happen **inside a pipeline** so that preprocessing is learned only from training data and applied consistently to future data.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Example feature groups
categorical_features = ["gender", "city"]
numerical_features = ["age", "income"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            SimpleImputer(strategy="median"),
            numerical_features
        )
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

print(model)



# 16. Encoding Decision Guide 🧠

```text
Categorical Feature
        │
        ▼
Does it have a natural order?
        │
   ┌────┴────┐
  YES        NO
   │          │
   ▼          ▼
Ordinal    How many
Encoding   categories?
              │
         ┌────┴────┐
        LOW       HIGH
         │          │
         ▼          ▼
     One-Hot    Frequency /
                Count /
                Target /
                Binary /
                Hash
```

### Golden Rules

1. **Nominal → usually One-Hot**
2. **Ordinal → Ordinal Encoding**
3. **Target → Label Encoding**
4. **High cardinality → consider alternatives to One-Hot**
5. **Target Encoding → prevent leakage**
6. **Linear models → understand multicollinearity**
7. **Always consider the model and business problem**



# 17. Complete Comparison Table

| Encoding | Multiple Columns | Preserves Order | Uses Target | High Cardinality |
|---|---:|---:|---:|---:|
| Label | ❌ | ❌ | ❌ | ❌ |
| Ordinal | ❌ | ✅ | ❌ | ❌ |
| One-Hot | ✅ | ❌ | ❌ | ❌ |
| Dummy | ✅ | ❌ | ❌ | ❌ |
| Frequency | ❌ | ❌ | ❌ | ✅ |
| Count | ❌ | ❌ | ❌ | ✅ |
| Target | ❌ | ❌ | ✅ | ✅ |
| Binary | ✅ | ❌ | ❌ | ✅ |
| Hash | ✅ | ❌ | ❌ | ✅ |
| WoE | ❌ | ❌ | ✅ | Sometimes |



# 18. Frequently Asked Interview Questions ⭐⭐⭐

### Q1. What is encoding?

**Answer:** Encoding converts categorical variables into numerical representations that machine-learning algorithms can process.

### Q2. Why is encoding required?

**Answer:** Most ML algorithms operate on numerical values rather than raw categorical strings.

### Q3. What is One-Hot Encoding?

**Answer:** It creates a separate binary column for each category.

### Q4. When should you use Ordinal Encoding?

**Answer:** When categories have a meaningful natural order, such as Low, Medium, and High.

### Q5. Why should Label Encoding not blindly be used for nominal features?

**Answer:** It can introduce an artificial numerical ordering that does not actually exist.

### Q6. One-Hot vs Label Encoding?

**Answer:** Label Encoding uses one integer-coded column; One-Hot Encoding creates separate binary columns and does not impose an ordinal relationship.

### Q7. What is the Dummy Variable Trap?

**Answer:** In a linear model, including all one-hot categories can create perfect multicollinearity because one category can be represented by the others.

### Q8. How do you handle high-cardinality categorical variables?

**Answer:** Consider frequency/count, target, binary, hashing, or rare-category grouping depending on the problem.

### Q9. What is Target Encoding?

**Answer:** It replaces a category with a target-derived statistic, commonly the target mean for that category.

### Q10. What is the biggest problem with Target Encoding?

**Answer:** Data leakage. The target information from validation/test data must not influence the encoding learned from training data.

### Q11. What is cardinality?

**Answer:** The number of unique categories in a categorical feature.

### Q12. Should we always use One-Hot Encoding?

**Answer:** No. The choice depends on category type, cardinality, model, dataset size, memory, and leakage considerations.



# 19. Interview Scenarios 🎯

### Scenario 1

**Feature:** `Color = Red, Blue, Green`

**Best common choice:** One-Hot Encoding

**Why?** No natural order.

---

### Scenario 2

**Feature:** `Satisfaction = Low, Medium, High`

**Best common choice:** Ordinal Encoding

**Why?** A meaningful ranking exists.

---

### Scenario 3

**Feature:** `City = 10,000 unique cities`

**Would you blindly use One-Hot?**

**No.** Consider frequency/count, target encoding with leakage control, binary encoding, hashing, or grouping rare categories.

---

### Scenario 4

**Target:** `Diagnosis = B, M`

A common approach:

```text
B → 0
M → 1
```

Then evaluate the classifier using metrics such as:

- Confusion Matrix
- Precision
- Recall
- F1-score
- ROC-AUC

---

### Scenario 5

**Question:** You performed target encoding before train-test split. Is that safe?

**Answer:** No. It can cause target leakage. Split first and learn the encoding from training data only.



# 20. Final Interview Cheat Sheet 🚀

| Situation | Think About |
|---|---|
| Nominal + few categories | **One-Hot** |
| Ordered categories | **Ordinal** |
| Target labels | **Label Encoding** |
| High cardinality | **Frequency / Count / Binary / Hash / Target** |
| Target-based encoding | **Leakage prevention** |
| Linear model | **Multicollinearity / Dummy Trap** |
| Production ML | **Pipeline + `handle_unknown="ignore"`** |

## ⭐ One-Line Interview Answer

> **The choice of encoding depends on whether the categorical variable is nominal or ordinal, its cardinality, the ML algorithm, dimensionality, and the risk of target leakage.**


In [ ]:
# Quick self-test
questions = {
    1: "City = Pune, Mumbai, Delhi → Which encoding?",
    2: "Satisfaction = Low, Medium, High → Which encoding?",
    3: "10,000 unique cities → Would you blindly use One-Hot?",
    4: "What is the biggest risk of Target Encoding?",
    5: "What problem can occur when all dummy columns are used in a linear model?"
}

for number, question in questions.items():
    print(f"Q{number}. {question}")
